# Multitemporal Analysis & Post-Classification Consistency 

This notebook runs **live Google Earth Engine** integration checks for each phase of the multitemporal plan using the sample data in `data/`:
- `data/area_of_interest.shp`
- `data/Tes_classification_scheme.csv`
- `data/training_points.shp` (class column: `ID`)
- `data/validation_samples - labeled.shp` (ground-truth column: `Is labeled`)

## 0. Setup & Earth Engine Initialization

In [7]:
import os
import sys
import ee
import geemap
import pandas as pd
import geopandas as gpd

# Ensure local src/ is importable when running from notebooks/
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
src_path = os.path.join(repo_root, 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import luma_ge

# Auto-initialize via auth/*.json (or fall back to manual auth)
ok = luma_ge.auto_initialize()
if not ok:
    luma_ge.authenticate_manually()

status = luma_ge.get_auth_status()
print('Auth status:', status)
assert status['initialized'], 'Earth Engine must be initialized before running live tests!'

DEBUG: Starting auto_initialize...
DEBUG: All environment variables containing 'GOOGLE': ['GOOGLE_APPLICATION_CREDENTIALS']
DEBUG: Checking environment variables...
DEBUG: GOOGLE_SERVICE_ACCOUNT_JSON exists: False
DEBUG: GOOGLE_APPLICATION_CREDENTIALS: c:\Users\AFahrezi\Documents\GitHub\luma-stack\auth\epistem-luma-launching-92541db9962d.json
DEBUG: Using GOOGLE_APPLICATION_CREDENTIALS: c:\Users\AFahrezi\Documents\GitHub\luma-stack\auth\epistem-luma-launching-92541db9962d.json
DEBUG: Found service account file: c:\Users\AFahrezi\Documents\GitHub\luma-stack\auth\epistem-luma-launching-92541db9962d.json
Earth Engine initialized with service account: c:\Users\AFahrezi\Documents\GitHub\luma-stack\auth\epistem-luma-launching-92541db9962d.json
Auth status: {'initialized': True, 'authenticated': True, 'project': None, 'user_info': None}


In [8]:
#Define Area of Interest (AOI)
#Kambera, NTT
aoi = ee.FeatureCollection('projects/ee-agilfahrezy60/assets/Sesi1_AOI_Kambera_NTT')
#classification scheme
from luma_ge.classification_scheme import LULC_Scheme_Manager
manager = LULC_Scheme_Manager()
#path to csv 
csv_path = "../data/MODUL2_Skema_Klasifikasi_Kambera.csv"
scheme_df = pd.read_csv(csv_path, sep=None, engine="python")
print("", scheme_df)
#Training data
roi = ee.FeatureCollection('projects/ee-agilfahrezy60/assets/Sesi1_Data_Latih_Kambera') 
#validation data
validation = ee.FeatureCollection('projects/ee-agilfahrezy60/assets/Sesi2_Data_Penguji_Akurasi')

    ID        Land Cover Class Color Palette
0   1      Hutan Lahan Kering       #006400
1   2          Hutan Mangrove       #759237
2   3            Kebun Campur       #d534ff
3   4           Padang Rumput       #A7FF71
4   5  Pertanian Lahan Kering       #FFC500
5   6                   Sawah       #FFFD00
6   7         Lahan Terbangun       #ffa9eb
7   8           Lahan Terbuka       #ff1f1f
8   9               Badan Air       #0000FF


---
## Phase 1 Live EE Tests: `imagery_correction.py` & `data_acquisition.py`
Tests:
1. **1A**: `Reflectance_Data.get_hls_optical_data()` (NASA HLS v2.0 `Fmask` + `0.0001` scaling)
2. **1B**: `BRDF_Correction` + `Topographic_Correction` on a live Landsat 8 scene
3. **1C**: `get_multitemporal_optical_data()` (`L5_SR` 2010 + `L8_SR` 2024 with Roy 2016 + optional BRDF) and `get_multitemporal_composites()` (dropping `AEROSOL` + CDF `histogram_match`)

In [9]:
# Test 1A: NASA HLS v2.0 Live Acquisition
from luma_ge.data_acquisition import Reflectance_Data, final_Image

rd = Reflectance_Data()
hls_col, hls_stats = rd.get_hls_optical_data(
    aoi=aoi,
    start_date='2025-01-01',
    end_date='2025-12-31',
    dataset='HLS',
    cloud_cover=30,
    verbose=True,
    compute_detailed_stats=True,
)
print('HLS filtered image count:', hls_stats['filtered_collection']['total_images'])
if hls_stats['filtered_collection']['total_images'] > 0:
    print('HLS first image bands:', hls_col.first().bandNames().getInfo())
    hls_comp = final_Image().get_temporal_composite(hls_col, aoi, verbose=False)
    print('HLS composite bands:', hls_comp['image'].bandNames().getInfo())

2026-11-16 17:50:38,184 - Reflectance_Data - INFO - ReflectanceData initialized.
2026-11-16 17:50:38,191 - Reflectance_Stats - INFO - Reflectance Stats initialized.
2026-11-16 17:50:44,736 - Reflectance_Data - INFO - Retrieved NASA Harmonized Landsat Sentinel-2 (HLSL30 + HLSS30 v2.0, 30m) for 2025-01-01 to 2025-12-31


HLS filtered image count: 250


2026-11-16 17:50:45,459 - final_Image - INFO - final_Image creation initialized.


HLS first image bands: ['BLUE', 'GREEN', 'RED', 'NIR', 'SWIR1', 'SWIR2']
HLS composite bands: ['BLUE', 'GREEN', 'RED', 'NIR', 'SWIR1', 'SWIR2']


In [10]:
# Test 1B: Live BRDF_Correction & Topographic_Correction (SCSc) on Landsat 8 SR
from luma_ge.imagery_correction import BRDF_Correction, Topographic_Correction

l8_col, _ = rd.get_optical_data(
    aoi=aoi,
    start_date='2025-01-01',
    end_date='2025-12-31',
    optical_data='L8_SR',
    cloud_cover=30,
    verbose=False,
    compute_detailed_stats=False,
)
sample_scene = ee.Image(l8_col.first())

brdf = BRDF_Correction()
topo = Topographic_Correction()

scene_brdf = brdf.correct_image(sample_scene, verbose=True)
scene_brdf_topo = topo.correct_image(scene_brdf, scale=30, verbose=True)

# Verify properties and median reflectance on server
raw_med = sample_scene.select(['RED', 'NIR']).reduceRegion(ee.Reducer.median(), aoi, 60).getInfo()
corr_med = scene_brdf_topo.select(['RED', 'NIR']).reduceRegion(ee.Reducer.median(), aoi, 60).getInfo()
print('Raw scene median (RED/NIR):', raw_med)
print('BRDF + Topo corrected median (RED/NIR):', corr_med)
print('system:time_start preserved:', scene_brdf_topo.get('system:time_start').getInfo() is not None)

2026-11-16 17:50:54,189 - Reflectance_Stats - INFO - Reflectance Stats initialized.
2026-11-16 17:50:54,192 - BRDF_Correction - INFO - BRDF_Correction initialised.
2026-11-16 17:50:54,193 - Topographic_Correction - INFO - Topographic_Correction initialised.
2026-11-16 17:50:54,193 - BRDF_Correction - INFO - Applying BRDF correction to bands: ['BLUE', 'GREEN', 'RED', 'NIR', 'SWIR1', 'SWIR2']
2026-11-16 17:50:54,198 - Topographic_Correction - INFO - Applying SCSc topographic correction (scale=30m, slope_threshold=5.0 deg)
2026-11-16 17:50:54,199 - Topographic_Correction - INFO - Note: scale=30m (Landsat default). Pass scale=10 for Sentinel-2 imagery.
2026-11-16 17:50:54,201 - Topographic_Correction - INFO - SCSc correction applied to single image.


Raw scene median (RED/NIR): {'NIR': 0.333953671172473, 'RED': 0.053700896177943984}
BRDF + Topo corrected median (RED/NIR): {'NIR': 0.3457234129876385, 'RED': 0.055688124584479755}
system:time_start preserved: True


In [11]:
# Test 1C: Cross-Sensor Multitemporal Acquisition (2010 L5_SR + 2025 L8_SR) + CDF Histogram Matching
periods_cfg = {
    'baseline': {'start_date': 2025, 'end_date': 2025, 'sensor': 'L8_SR', 'cloud_cover': 30},
    'historical': {'start_date': 2010, 'end_date': 2010, 'sensor': 'L5_SR', 'cloud_cover': 60},
}

mt_data = rd.get_multitemporal_optical_data(
    aoi=aoi,
    periods=periods_cfg,
    harmonize_landsat=True,
    apply_brdf=True,
    apply_topo_correction=False,
    verbose=True,
    compute_detailed_stats=True,
)
print('Chronological order:', mt_data['period_order'])

fi = final_Image()
mt_composites = fi.get_multitemporal_composites(
    collections_by_period=mt_data,
    aoi=aoi,
    reference_period='baseline',
    reducer='median',
    drop_aerosol=True,
    harmonize_sensors=True,
    histogram_match=True,
    scale=30,
    max_buckets=128,
    verbose=True,
)
print('Common bands (AEROSOL dropped):', mt_composites['common_bands'])

# Compare median reflectance before vs after histogram matching
geom = aoi.geometry() if isinstance(aoi, ee.FeatureCollection) else aoi
raw_hist_med = mt_composites['raw_composites']['historical'].select(['BLUE', 'RED', 'NIR']).reduceRegion(ee.Reducer.median(), geom, 60).getInfo()
matched_hist_med = mt_composites['composites']['historical'].select(['BLUE', 'RED', 'NIR']).reduceRegion(ee.Reducer.median(), geom, 60).getInfo()
base_med = mt_composites['composites']['baseline'].select(['BLUE', 'RED', 'NIR']).reduceRegion(ee.Reducer.median(), geom, 60).getInfo()
print('Historical RAW median:', raw_hist_med)
print('Historical MATCHED median:', matched_hist_med)
print('Baseline (reference) median:', base_med)

Map1 = geemap.Map()
vis_rgb = {'bands': ['RED', 'GREEN', 'BLUE'], 'min': 0.0, 'max': 0.25, 'gamma': 1.3}
Map1.addLayer(mt_composites['raw_composites']['historical'], vis_rgb, '2010 Historical (Raw)')
Map1.addLayer(mt_composites['composites']['historical'], vis_rgb, '2010 Historical (Histogram Matched)')
Map1.addLayer(mt_composites['composites']['baseline'], vis_rgb, '2024 Baseline')
Map1.centerObject(aoi, 12)
Map1

2026-11-16 17:50:58,478 - BRDF_Correction - INFO - BRDF_Correction initialised.
2026-11-16 17:50:58,479 - Reflectance_Data - INFO - Starting data fetch for Landsat 5 Thematic Mapper Surface Reflectance Collection
2026-11-16 17:50:58,479 - Reflectance_Data - INFO - Date range: 2010-01-01 to 2010-12-31
2026-11-16 17:50:58,479 - Reflectance_Data - INFO - Cloud cover threshold: 60%
2026-11-16 17:50:58,479 - Reflectance_Stats - INFO - Reflectance Stats initialized.
2026-11-16 17:50:59,463 - Reflectance_Data - INFO - Initial collection (before cloud filtering): 7 images
2026-11-16 17:50:59,464 - Reflectance_Data - INFO - Date range of available images: 2010-01-22 to 2010-12-24
2026-11-16 17:51:00,356 - Reflectance_Data - INFO - After cloud filtering (<60%): 4 images
2026-11-16 17:51:00,356 - Reflectance_Data - INFO - Cloud cover of selected images: 40.0% - 51.0%
2026-11-16 17:51:00,356 - Reflectance_Data - INFO - Average cloud cover: 46.0%
2026-11-16 17:51:00,358 - Reflectance_Data - INFO - 

Chronological order: ['historical', 'baseline']


2026-11-16 17:51:04,943 - final_Image - INFO - Creating median composite from 4 images
2026-11-16 17:51:04,944 - final_Image - INFO - Composite clipped to AOI
2026-11-16 17:51:06,194 - final_Image - INFO - Composite created from 2010-01-22 to 2010-12-08
2026-11-16 17:51:06,813 - final_Image - INFO - Creating median composite from 9 images
2026-11-16 17:51:06,814 - final_Image - INFO - Composite clipped to AOI
2026-11-16 17:51:08,467 - final_Image - INFO - Composite created from 2025-02-16 to 2025-09-28


Common bands (AEROSOL dropped): ['BLUE', 'GREEN', 'RED', 'NIR', 'SWIR1', 'SWIR2']
Historical RAW median: {'BLUE': 0.06298862020263679, 'NIR': 0.29390240278443536, 'RED': 0.0893496943385156}
Historical MATCHED median: {'BLUE': 0.040051564943596386, 'NIR': 0.270524716473897, 'RED': 0.06922657725597334}
Baseline (reference) median: {'BLUE': 0.04150803771949001, 'NIR': 0.2705058614356431, 'RED': 0.07131872607846824}


Map(center=[-9.704356522667766, 120.27216753241345], controls=(WidgetControl(options=['position', 'transparent…

---
## Phase 2 Live EE Tests: `sample_data.py` & `sample_data_quality.py`
Tests:
1. **2A**: `SyncTrainData.LoadMultitemporalTrainData` + `SplitTrainData.SplitMultitemporalProcess` (1 dataset → Model Transfer routing)
2. **2B**: `sample_quality.run_multitemporal_separability` (live GEE pixel extraction + TD distance on `baseline` only, skipping `historical`)

In [12]:
from luma_ge.sample_data import SyncTrainData, SplitTrainData
from luma_ge.sample_data_quality import sample_quality

# Kambera EE FeatureCollection uses 'ID_Kelas' (class ID) and 'KelasTupla' (class name)
CLASS_ID_PROP = 'ID_Kelas'
CLASS_NAME_PROP = 'KelasTupla'

train_loaded = SyncTrainData.LoadMultitemporalTrainData(
    training_sources={'historical': None, 'baseline': roi},
    landcover_df=scheme_df,
    aoi_geometry=aoi,
    period_order=mt_composites['period_order'],
    reference_period='baseline',
    class_field=CLASS_ID_PROP,
    use_class_ids=True,
    min_samples=5,
)
print('Detected mode:', train_loaded['mode'])
print('Validation summary:', train_loaded['validation_summary'])

# Works seamlessly whether training_data is a GeoDataFrame or an ee.FeatureCollection
splits = SplitTrainData.SplitMultitemporalProcess(
    train_loaded['training_by_period'],
    TrainSplitPct=0.7,
    random_state=42,
    class_field=CLASS_ID_PROP,
)
train_split_obj, test_split_obj = splits['baseline']
if isinstance(train_split_obj, ee.FeatureCollection):
    print('Split counts (EE train, test):', train_split_obj.size().getInfo(), test_split_obj.size().getInfo())
else:
    print('Split counts (GDF train, test):', len(train_split_obj), len(test_split_obj))

# Live GEE separability check (Model Transfer skips historical)
sep_res = sample_quality.run_multitemporal_separability(
    composites_by_period=mt_composites,
    training_by_period=train_loaded['training_by_period'],
    class_property=CLASS_ID_PROP,
    region=geom,
    class_name_property=CLASS_NAME_PROP,
    scale=30,
    max_pixels_per_class=5000,
)
print('Analyzed periods:', sep_res['analyzed_periods'])
print('Skipped periods:', sep_res['skipped_periods'])
display(sep_res['per_period']['baseline']['summary_df'])
display(sep_res['per_period']['baseline']['separability_df'])

2026-11-16 17:52:15,790 - luma_ge.sample_data - INFO - Validating classes with use_class_ids=True
2026-11-16 17:52:15,792 - luma_ge.sample_data - INFO - Class field: ID_Kelas
2026-11-16 17:52:15,792 - luma_ge.sample_data - INFO - Training data type: <class 'ee.featurecollection.FeatureCollection'>
2026-11-16 17:52:15,792 - luma_ge.sample_data - INFO - Landcover DF columns: ['ID', 'Land Cover Class', 'Color Palette']
2026-11-16 17:52:15,792 - luma_ge.sample_data - INFO - Valid IDs in landcover_df: [1, 2, 3, 4, 5, 6, 7, 8, 9]
2026-11-16 17:52:15,794 - luma_ge.sample_data - INFO - Processing Earth Engine FeatureCollection
2026-11-16 17:52:17,139 - luma_ge.sample_data - INFO - Unique classes in EE FeatureCollection: [9, 1, 2, 3, 7, 8, 4, 5, 6]
2026-11-16 17:52:17,140 - luma_ge.sample_data - INFO - Valid classes: [9, 1, 2, 3, 7, 8, 4, 5, 6]
2026-11-16 17:52:17,140 - luma_ge.sample_data - INFO - Invalid classes: []
2026-11-16 17:52:18,552 - luma_ge.sample_data - INFO - Features after class v

Detected mode: model_transfer
Validation summary: {'baseline': {'total_points': 460, 'valid_points': 460, 'points_after_class_filter': 460, 'invalid_classes': [], 'outside_aoi': [], 'insufficient_samples': [], 'warnings': []}}
Split counts (GDF train, test): 0 0
Extracted spectral values for 459 samples across 9 classes
Analyzed periods: ['baseline']
Skipped periods: ['historical']


,Total Pairs,Good Separability Pairs,Weak Separability Pairs,Poor Separability Pairs
0,36,34,2,0


,Class1_ID,Class1_Name,Class2_ID,Class2_Name,TD_Distance,Separability_Level
0,3,Kebun Campur,5,Pertanian Lahan Kering,1.389,🟡 Weak/Marginal Separability (1.0 ≤ TD < 1.8)
1,4,Padang Rumput,5,Pertanian Lahan Kering,1.647,🟡 Weak/Marginal Separability (1.0 ≤ TD < 1.8)
2,7,Lahan Terbangun,8,Lahan Terbuka,1.836,🟢 Good Separability (TD ≥ 1.8)
3,5,Pertanian Lahan Kering,6,Sawah,1.907,🟢 Good Separability (TD ≥ 1.8)
4,3,Kebun Campur,6,Sawah,1.927,🟢 Good Separability (TD ≥ 1.8)
5,1,Hutan Lahan Kering,3,Kebun Campur,1.936,🟢 Good Separability (TD ≥ 1.8)
6,7,Lahan Terbangun,5,Pertanian Lahan Kering,1.958,🟢 Good Separability (TD ≥ 1.8)
7,9,Badan Air,6,Sawah,1.969,🟢 Good Separability (TD ≥ 1.8)
8,3,Kebun Campur,7,Lahan Terbangun,1.970,🟢 Good Separability (TD ≥ 1.8)
9,3,Kebun Campur,4,Padang Rumput,1.990,🟢 Good Separability (TD ≥ 1.8)


---
## Phase 3 Live EE Tests: Early Phase Predictor (`multitemporal.py`)
Verifies that predictors (`common_bands` + terrain + compatible spectral indices) are computed on **both** `historical` and `baseline` with identical band lists.

In [13]:
from luma_ge.multitemporal import compute_multitemporal_predictors

predictor_config = {
    'individual_predictors': {
        'elevation': True,
        'slope': True,
        'aspect': False,
        'spectral_indices': ['NDVI', 'MNDWI', 'NDBI'],
    }
}

pred_res = compute_multitemporal_predictors(
    composites_by_period=mt_composites,
    aoi=geom,
    predictor_config=predictor_config,
    collections_by_period=mt_data,
    mode='model_transfer',
)
print('Compatible indices:', pred_res['compatible_indices'])
print('Skipped indices:', pred_res['skipped_indices'])
print('Common feature bands:', pred_res['common_feature_bands'])
for p, stack in pred_res['predictor_stacks'].items():
    print(f'Live EE bands for {p}:', stack.bandNames().getInfo())

2026-11-16 17:53:43,714 - luma_ge.predictor - INFO - SpectralCalculator initialized
2026-11-16 17:53:43,714 - luma_ge.predictor - INFO - Terrain calculator initialized
2026-11-16 17:53:43,715 - luma_ge.predictor - INFO - Calculating elevation layer using NASADEM DEM...
2026-11-16 17:53:43,716 - luma_ge.predictor - INFO - Successfully calculated elevation layer using NASADEM DEM
2026-11-16 17:53:43,716 - luma_ge.predictor - INFO - Calculating slope layer using NASADEM DEM...
2026-11-16 17:53:43,716 - luma_ge.predictor - INFO - Calculating elevation layer using NASADEM DEM...
2026-11-16 17:53:43,717 - luma_ge.predictor - INFO - Successfully calculated elevation layer using NASADEM DEM
2026-11-16 17:53:43,717 - luma_ge.predictor - INFO - Successfully calculated slope layer using NASADEM DEM


Compatible indices: ['NDVI', 'MNDWI', 'NDBI']
Skipped indices: []
Common feature bands: ['BLUE', 'GREEN', 'RED', 'NIR', 'SWIR1', 'SWIR2', 'elevation', 'slope', 'NDVI', 'MNDWI', 'NDBI']
Live EE bands for historical: ['BLUE', 'GREEN', 'RED', 'NIR', 'SWIR1', 'SWIR2', 'elevation', 'slope', 'NDVI', 'MNDWI', 'NDBI']
Live EE bands for baseline: ['BLUE', 'GREEN', 'RED', 'NIR', 'SWIR1', 'SWIR2', 'elevation', 'slope', 'NDVI', 'MNDWI', 'NDBI']


---
## Phase 4 Live EE Tests: `MULTIPROBABILITY` Model Transfer (`classification.py`) & Accuracy (`accuracy.py`)

In [14]:
from luma_ge.classification import FeatureExtraction, Generate_LULC
from luma_ge.input_utils import EE_converter

train_split_obj, test_split_obj = splits['baseline']
if isinstance(train_split_obj, ee.FeatureCollection):
    train_ee = train_split_obj
    test_ee = test_split_obj
else:
    converter = EE_converter(verbose=False)
    train_ee = converter.convert_roi_gdf(train_split_obj[[CLASS_ID_PROP, 'geometry']])
    test_ee = converter.convert_roi_gdf(test_split_obj[[CLASS_ID_PROP, 'geometry']])

ref_stack = pred_res['predictor_stacks']['baseline'].select(pred_res['common_feature_bands'])
train_sampled = ref_stack.sampleRegions(collection=train_ee, properties=[CLASS_ID_PROP], scale=30, tileScale=4)
test_sampled = ref_stack.sampleRegions(collection=test_ee, properties=[CLASS_ID_PROP], scale=30, tileScale=4)

lulc = Generate_LULC()
clf_res = lulc.classify_multitemporal(
    predictor_stacks=pred_res['predictor_stacks'],
    training_splits={'baseline': (train_sampled, test_sampled)},
    class_property=CLASS_ID_PROP,
    mode='model_transfer',
    reference_period='baseline',
    classification_type='soft',
    soft_mode='multiprobability',
    ntrees=100,
    common_bands=pred_res['common_feature_bands'],
    period_order=pred_res['period_order'],
)
print('Mode used:', clf_res['mode_used'])
print('Class IDs:', clf_res['class_ids'])
print('Baseline model eval OA:', clf_res['model_evaluation']['baseline']['overall_accuracy'])
print('Baseline model eval Kappa:', clf_res['model_evaluation']['baseline']['kappa'])
display(clf_res['feature_importance']['baseline'].head(10))

for p, m in clf_res['maps'].items():
    print(f'Classified map bands ({p}):', m.bandNames().getInfo())

Map4 = geemap.Map()
vis_lulc = {
    'bands': ['classification'],
    'min': int(scheme_df['ID'].min()),
    'max': int(scheme_df['ID'].max()),
    'palette': scheme_df.sort_values('ID')['Color Palette'].tolist(),
}
Map4.addLayer(clf_res['maps']['historical'].clip(geom), vis_lulc, '2010 Historical LULC (Transferred)')
Map4.addLayer(clf_res['maps']['baseline'].clip(geom), vis_lulc, '2025 Baseline LULC')
Map4.addLayer(clf_res['maps']['historical'].select('confidence').clip(geom), {'min': 25, 'max': 100}, '2010 Transfer Confidence (%)', False)
Map4.centerObject(aoi, 12)
Map4

KeyError: "None of [Index(['ID_Kelas', 'geometry'], dtype='object')] are in the [columns]"

In [ ]:
from luma_ge.accuracy import thematic_accuracy

# Auto-detect ground truth class property on the validation FeatureCollection
val_props = validation.first().propertyNames().getInfo()
print('Validation asset properties:', val_props)
val_class_prop = next(
    (c for c in ('ID_Kelas', 'class_id', 'Is labeled', 'kelas', 'ID') if c in val_props),
    val_props[0],
)
print('Using validation class property:', val_class_prop)

taa = thematic_accuracy()
acc_res = taa.run_multitemporal_assessment(
    maps_by_period=clf_res['maps'],
    validation_by_period={'baseline': validation, 'historical': None},
    class_property=val_class_prop,
    scale=30,
    required_periods=['baseline'],
    period_order=clf_res['period_order'],
)
print('Assessed periods:', acc_res['assessed_periods'])
print('Skipped periods:', acc_res['skipped_periods'])
ok_base, base_metrics = acc_res['per_period']['baseline']
print('Baseline Thematic Accuracy Summary:', taa.format_accuracy_summary(base_metrics) if ok_base else base_metrics)

---
## Phase 5 & Phase 6 Live EE Tests (Pending Implementation)
Once Phases 5 (`MultitemporalAnalysis` wrapper) and 6 (`ConsistencyAnalyzer` in `post_classification.py`) are implemented, live GEE test cells for end-to-end orchestration, cross-tabulation, and HMM/Viterbi vs. rule-based consistency correction will be appended here.